# Proteomics
Extract timepoints from dataset and create two new datasets from it
Creates new *mid_processing_datasets* folder and parquet files during runtime


In [ ]:
from src.common_imports import *
from src.data_audit import data_audit
from src.somascan import *
from src.save_wrapper import save_report
from src.extraction import extract_dtypes
from src.check_data import *
from src.normalization import *
from src.low_variance_report import low_variance_report


In [ ]:
data = load()

In [ ]:
print("Indexing data...")
df_exp = data['df_expMatrix']
df_samp = data['df_sampMatrix']
print("Done!")

# Data and low variance reports
Shows an overview of the following
data_audit:
-   Dataframe shape ( rows x columns)
-   Duplicates
-   Missing Values
-   Numeric Features
-   Outliers by IQR and 3σ(68-95-99.7)
-   Categorical Consistency
-   Cleaning Summary

low_variance_report:
-   Total numeric features
-   predefined low_variance_threshold
-   low variance features found

In [ ]:
# data analysis of raw data
save_report(data_audit, df_exp, "expression matrix")
save_report(data_audit,df_samp, "sample matrix")
save_report(low_variance_report, df_exp, "expression matrix")
save_report(low_variance_report, df_samp, "sample matrix")

# Extracting and saving
After taking a look at the data we extract them by timepoint
and split into two new datasets
additionally we create a pseudo glossary for if one wants to lookup the gene

In [ ]:
# save splits
splits = reshape_and_split(df_exp, df_samp)

# save splits
save_splits(splits)

# create glossary
create_gene_lookup(df_exp)

# Verifiying
After saving the files we verify it by pulling from the new source
and running the data_audit and low_variance_report on them

In [ ]:
# pulling saved data
df_exp_bl = pd.read_parquet("../../mid_processing_datasets/expression_matrix_baseline.parquet")
df_exp_6m = pd.read_parquet("../../mid_processing_datasets/expression_matrix_6month.parquet")
print("done reading")

In [ ]:
# running data audit on newly formed data
save_report(data_audit, df_exp_bl, "baseline expression matrix")
save_report(data_audit, df_exp_6m, "6 months expression matrix")
save_report(low_variance_report,df_exp_bl, "baseline expression matrix")
save_report(low_variance_report, df_exp_6m, "6 months expression matrix")

In [ ]:
# further sample verificiation
display(df_exp_bl.head())
display(df_exp_6m.head())

In [ ]:
extract_dtypes(df_exp_bl, verbose=True)

In [ ]:
#show_sheet_overview(df_exp_bl)
#show_dataset_structure(df_exp_bl)
inspect_omics_dataset(df_exp_bl)

In [2]:
import pandas as pd
gene_lookup = pd.read_parquet('../../mid_processing_datasets/gene_lookup.parquet')
gene_lookup.loc['7660-21_3']

EntrezGeneSymbol                         TPM4
EntrezGeneID                             7171
Target                          Tropomyosin 4
TargetFullName      Tropomyosin alpha-4 chain
UniProt                                P67936
Name: 7660-21_3, dtype: str

In [ ]:
normalize_num_dtype(df_exp_bl)
df_exp_bl.dtypes
normalize_num_dtype(df_exp_6m)
df_exp_6m.dtpyes

# Summary
After running this notebook the preprocessing has been done and the data is ready to merge or be analyzed further
## Understanding Data
RFU, SeqId, SomaId